In [2]:
import params, os, funcs
from jaratoolbox import celldatabase, settings
import numpy as np

# Load dataframe
databaseDir = os.path.join(settings.DATABASE_PATH, '2024popanalysis')
fullDbPath = 'celldb_2024popanalysis.h5'
fullPath = os.path.join(databaseDir, fullDbPath)
fullDb = celldatabase.load_hdf(fullPath)
simpleSiteNames = fullDb["recordingSiteName"].str.split(',').apply(lambda x: x[0])
simpleSiteNames = simpleSiteNames.replace("Posterior auditory area", "Dorsal auditory area")
fullDb["recordingSiteName"] = simpleSiteNames
fullDb.head()

/Volumes/NardociData/jarahubdata/figuresdata/2024popanalysis/celldb_2024popanalysis.h5 does not exist or cannot be opened.


FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '/Volumes/NardociData/jarahubdata/figuresdata/2024popanalysis/celldb_2024popanalysis.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
X_speech_all, Y_brain_area_speech_all, X_AM_all = {}, {}, {}
Y_brain_area_AM_all, X_pureTones_all, Y_brain_area_PT_all = {}, {}, {}
previous_frequency_speech, previous_frequency_AM, previous_frequency_PT = None, None, None
indices_AM, indices_PT, indices_speech = None, None, None

# Initialize dictionaries for each spike window
for window_name in params.spike_windows:
    if "speech" in window_name:
        X_speech_all[window_name] = []
        Y_brain_area_speech_all[window_name] = []
    if "am" in window_name:
        X_AM_all[window_name] = []
        Y_brain_area_AM_all[window_name] = []
    if "pt" in window_name:
        X_pureTones_all[window_name] = []
        Y_brain_area_PT_all[window_name] = []

In [ ]:
def spike_rate(sound_type:str, ensemble, ephysData, bdata, targetSiteName):
    '''
    Calculate firing rates for multiple evoked spike windows.

    Returns a dictionary of spikeRateNormalized arrays keyed by window name.
    Each value is (trials, neurons).
    
    sound_type = ['speech', 'pt', 'am']
    '''
    X_dict = {}
    Y_brain_area_array = []

    if sound_type == "speech":
        eventOnsetTimes = ephysData['events']['stimOn']
        FTParamsEachTrial = bdata['targetFTpercent']
        VOTParamsEachTrial = bdata['targetVOTpercent']
        Y_frequency = np.array([(FTParamsEachTrial[i], VOTParamsEachTrial[i]) for i in range(len(FTParamsEachTrial))])

    else:  # AM or PT
        eventOnsetTimes = ephysData['events']['stimOn'][:len(bdata['currentFreq'])]
        Y_frequency = np.array(bdata['currentFreq'])

    # Loop over relevant windows for this sound_type
    for label, (start, end) in params.spike_windows.items():
        if sound_type in label:
            _, _, _ = ensemble.eventlocked_spiketimes(eventOnsetTimes, [start, end])
            spikeCounts = ensemble.spiketimes_to_spikecounts(np.arange(start, end, 0.01))  # Bin width of 10 ms
            sumEvokedFR = spikeCounts.sum(axis=2)
            spikesPerSecEvoked = sumEvokedFR / (end - start)

            trialMeans = spikesPerSecEvoked.mean(axis=1)
            spikesPerSecEvokedNormalized = spikesPerSecEvoked.T - trialMeans
            spikesPerSecEvokedNormalized = spikesPerSecEvokedNormalized.T

            if spikesPerSecEvokedNormalized.shape[1] > params.leastCellsArea:
                subsetIndex = np.random.choice(spikesPerSecEvokedNormalized.shape[1], params.leastCellsArea, replace=False)
                spikeRateNormalized = spikesPerSecEvokedNormalized[:, subsetIndex]
            else:
                spikeRateNormalized = spikesPerSecEvokedNormalized

            X_dict[label] = spikeRateNormalized
            Y_brain_area_array = [targetSiteName] * spikeRateNormalized.shape[0]

    return X_dict, Y_brain_area_array, Y_frequency

In [ ]:
# Add data to the dictionary for each brain area and sound type
for subject in params.subject_list:
    for date in params.recordingDate_list[subject]:
        for brain_area in params.targetSiteNames:
            speechEnsemble, speechEphys, speechBdata = funcs.load_data(subject, date, brain_area, "FTVOTBorders")

            if speechEnsemble:
                X_speech_dict, Y_brain_area_speech, Y_frequency_speech = spike_rate("speech", speechEnsemble, speechEphys, speechBdata, brain_area)
                for window_name, X_speech in X_speech_dict.items():
                    X_speech_array, Y_frequency_speech_sorted, previous_frequency_speech, indices_speech = (
                        funcs.adjust_speech_length(subject, date, brain_area, X_speech, Y_frequency_speech,
                                                   Y_brain_area_speech, previous_frequency_speech))
                    
                    if X_speech_array is not None:
                        Y_frequency_FT = Y_frequency_speech_sorted[:, 0]
                        Y_frequency_VOT = Y_frequency_speech_sorted[:, 1]
                
                        X_speech_all[window_name].extend(X_speech_array)
                        Y_brain_area_speech_all[window_name].extend(Y_brain_area_speech)

            # Load and process data for AM
            amEnsemble, amEphys, amBdata = funcs.load_data(subject, date, brain_area, "AM")
            if amEnsemble:
                X_AM_dict, Y_brain_area_AM, Y_frequency_AM = spike_rate("am", amEnsemble, amEphys, amBdata, 
                                                                        brain_area)
                
                for window_name, X_AM in X_AM_dict.items():
                    # Apply adjustments
                    X_AM_adjusted, Y_frequency_AM_adjusted, Yba_AM_adj = (
                        funcs.adjust_array_and_labels(X_AM, Y_frequency_AM, Y_brain_area_AM, 
                                                      params.max_trials['AM'], subject, date, brain_area))
                
                    # Sort arrays
                    X_AM_array, Y_frequency_AM_sorted, Y_brain_area_AM_sorted, previous_frequency_AM, indices_AM = (
                        funcs.sort_sound_array(subject, date, brain_area, X_AM_adjusted, Yba_AM_adj, Y_frequency_AM_adjusted, previous_frequency_AM))
                
                    X_AM_all[window_name].extend(X_AM_array)
                    Y_brain_area_AM_all[window_name].extend(Y_brain_area_AM_sorted)
                        
            # Load and process data for PT
            ptEnsemble, ptEphys, ptBdata = funcs.load_data(subject, date, brain_area, "pureTones")
            if ptEnsemble:
                X_PT_dict, Y_brain_area_PT, Y_frequency_pureTones = spike_rate("pt", ptEnsemble, ptEphys, ptBdata, 
                                                                        brain_area)
                for window_name, X_PT in X_PT_dict.items():
                    # Apply adjustments
                    X_PT_adjusted, Y_frequency_PT_adjusted, Yba_PT_adj = (
                        funcs.adjust_array_and_labels(X_PT, Y_frequency_pureTones, Y_brain_area_PT, 
                                                      params.max_trials['PT'], subject, date, brain_area))
                
                    # Sort arrays
                    X_PT_array, Y_frequency_PT_sorted, Y_brain_area_PT_sorted, previous_frequency_PT, indices_PT = (
                        funcs.sort_sound_array(subject, date, brain_area, X_PT_adjusted, Yba_PT_adj, 
                                               Y_frequency_PT_adjusted, previous_frequency_PT))
                
                    if X_PT_array is not None:
                        X_pureTones_all[window_name].extend(X_PT_array)
                        Y_brain_area_PT_all[window_name].extend(Y_brain_area_PT_sorted)

In [ ]:
# AM
X_AM_dict = {}
Y_brain_area_AM_all_renamed = {}

for window_name, X_AM_list in X_AM_all.items():
    simple_name = params.window_name_mapping.get(window_name, window_name)
    X_AM_sorted = funcs.sort_x_arrays(X_AM_list, indices_AM, "am")
    X_AM_array = np.stack(X_AM_sorted, axis=0)
    X_AM_dict[simple_name] = X_AM_array

    if window_name in Y_brain_area_AM_all:
        Y_brain_area_AM_all_renamed[simple_name] = Y_brain_area_AM_all[window_name]

# PT
X_PT_dict = {}
Y_brain_area_PT_all_renamed = {}

for window_name, X_PT_list in X_pureTones_all.items():
    simple_name = params.window_name_mapping.get(window_name, window_name)
    X_PT_sorted = funcs.sort_x_arrays(X_PT_list, indices_PT, "pt")
    X_PT_array = np.stack(X_PT_sorted, axis=0)
    X_PT_dict[simple_name] = X_PT_array

    if window_name in Y_brain_area_PT_all:
        Y_brain_area_PT_all_renamed[simple_name] = Y_brain_area_PT_all[window_name]

# Speech
X_speech_dict = {}
Y_brain_area_speech_all_renamed = {}

for window_name, X_speech_list in X_speech_all.items():
    simple_name = params.window_name_mapping.get(window_name, window_name)
    X_speech_sorted = funcs.sort_x_arrays(X_speech_list, indices_speech, "speech")
    X_speech_array = np.stack(X_speech_sorted, axis=0)
    X_speech_dict[simple_name] = X_speech_array

    if window_name in Y_brain_area_speech_all:
        Y_brain_area_speech_all_renamed[simple_name] = Y_brain_area_speech_all[window_name]

In [ ]:
print(f"AM Onset Shape: {X_AM_dict['onset'].shape}")
print(f"AM Sustained Shape: {X_AM_dict['sustained'].shape}")
print(f"AM Offset Shape: {X_AM_dict['offset'].shape}")

print(f"PT Onset Shape: {X_PT_dict['onset'].shape}")
print(f"PT Sustained Shape: {X_PT_dict['sustained'].shape}")
print(f"PT Offset Shape: {X_PT_dict['offset'].shape}")

print(f"Speech Onset Shape: {X_speech_dict['onset'].shape}")
print(f"Speech Sustained Shape: {X_speech_dict['sustained'].shape}")
print(f"Speech Offset Shape: {X_speech_dict['offset'].shape}")

In [1]:
data_dict = {}

for brain_area in params.targetSiteNames:
    for window_name in X_speech_dict:
        brain_area_mask = np.array(Y_brain_area_speech_all_renamed[window_name]) == brain_area
        X_array = X_speech_dict[window_name][brain_area_mask].T
        data_dict[(brain_area, 'FT', window_name)] = {'X': X_array, 'Y': Y_frequency_FT}
        data_dict[(brain_area, 'VOT', window_name)] = {'X': X_array, 'Y': Y_frequency_VOT}

    for window_name in X_AM_dict:
        brain_area_mask = np.array(Y_brain_area_AM_all_renamed[window_name]) == brain_area
        X_array = X_AM_dict[window_name][brain_area_mask].T
        data_dict[(brain_area, 'AM', window_name)] = {'X': X_array, 'Y': Y_frequency_AM_sorted}

    for window_name in X_PT_dict:
        brain_area_mask = np.array(Y_brain_area_PT_all_renamed[window_name]) == brain_area
        X_array = X_PT_dict[window_name][brain_area_mask].T
        data_dict[(brain_area, 'PT', window_name)] = {'X': X_array, 'Y': Y_frequency_PT_sorted}
        
data_dict[('Ventral auditory area', 'VOT', 'onset')]['X'].shape

NameError: name 'params' is not defined